## IMPORTS


In [ ]:
import pandas as pd
import numpy as np
import optuna
import os
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import sys
ruta_raiz = os.path.abspath('..')
if ruta_raiz not in sys.path:
    sys.path.append(ruta_raiz)

%load_ext autoreload
%autoreload 2
from Data_preprocesing.IQRCapper import IQRCapper



¡IQRCapper importado con éxito!


## LOAD DATA + SAVE FUCTION


In [8]:
X_train = pd.read_csv("../Train_test_split/X_train.csv")
y_train = pd.read_csv("../Train_test_split/y_train.csv")
X_test = pd.read_csv("../Train_test_split/X_test.csv")
y_test = pd.read_csv("../Train_test_split/y_test.csv")



target_folder = '../comparations'

def save_experiment(model_name, imbalance_method, accuracy, precision, recall, f1, roc_auc):
    new_result = {
        'Model': [model_name],
        'Imbalance_Method': [imbalance_method],
        'Accuracy': [round(accuracy, 4)],
        'Precision': [round(precision, 4)],
        'Recall': [round(recall, 4)],
        'F1_Score': [round(f1, 4)],
        'ROC_AUC': [round(roc_auc, 4)]
    }
    df_new = pd.DataFrame(new_result)
    
    # Point the file to inside the folder
    csv_file = f'{target_folder}/imbalance_results.csv'
    
    if os.path.exists(csv_file):
        df_new.to_csv(csv_file, mode='a', header=False, index=False)
    else:
        df_new.to_csv(csv_file, index=False)
        
    print(f"✅ Success! Results for {model_name} + {imbalance_method} saved in the '{target_folder}' folder.")


## XGBOOST

In [25]:
# Vamos a dejar los splits en el formato necesario
train = xgb.DMatrix(data=X_train, label=y_train)
test  = xgb.DMatrix(data=X_test, label=y_test)

#vamos a definir la variable objetivo
num_classes = int(y_train['customer_segment'].nunique())

learning_objective = {
    'objective': 'multi:softmax', 
    'num_class': num_classes,
    'eval_metric': 'mlogloss'
}

# Creamos un modelo "basico para confirmar que el modelo es util para nuestro caso"
model = xgb.train(params = learning_objective, dtrain= train)
test_predictions = model.predict(test)
print(accuracy_score(y_test, test_predictions))
print(f1_score(y_test, test_predictions, average='weighted'))


0.7607
0.7569068550318553


Con un modelo basico sin ningun tipo de ajuste de hyperparametros obtenemos un accuracy de 0.76 y un f1 score de 0.75(buena señal). Viedno los resultados decidimos seguir con el siguiente paso: Optuna (el metedo de balanceo lo decidiremos como si fuse un hyperparametro de optuna)

In [ ]:
imbalance_methods = {
    "Baseline": None,
    "RandomUnderSampler":     RandomUnderSampler(random_state=42),
    "TomekLinks":             TomekLinks(),
    "RandomOverSampler":      RandomOverSampler(random_state=42),
    "SMOTE":                  SMOTE(random_state=42),
    "ADASYN":                 ADASYN(random_state=42),
    "SMOTETomek":             SMOTETomek(random_state=42),
    "SMOTEENN":               SMOTEENN(random_state=42),
}